# Learning curve `g_80x30` — thêm dữ liệu có giúp không?

Train cùng một nhóm trên **250 / 500 / 1000 / toàn bộ 1562** ảnh, đo tất cả trên
**cùng một tập test giữ riêng**, nên 4 điểm so sánh được với nhau.

Câu trả lời nằm ở **đoạn cuối đường cong**:

- Còn leo dốc ở 1562 → thêm ảnh cùng nguồn sẽ tăng accuracy.
- Đã phẳng, trong khi nhãn vẫn còn lỗi đã biết (font vẽ số `1` kiểu serif bị gán
  thành `I`/`l`, model đọc đúng `0691` còn tên file ghi `069I`) → trần là do **gán
  nhãn**, thêm ảnh cùng nguồn không dịch được nó.

## Vì sao cố định *step* chứ không cố định *epoch*

Batch 64: điểm 250 ảnh chạy 4 step/epoch, điểm 1562 chạy 25 step/epoch. Giữ 25
epoch cho mọi điểm nghĩa là điểm nhỏ nhất chỉ được **100 step** còn điểm lớn nhất
**625 step** — chênh 6×. Đường cong khi đó đo *lượng train*, không đo *lượng dữ liệu*.

Đây là số đo, không phải suy đoán: lần chạy đầu dùng 25 epoch cố định và điểm
`n=250` kết thúc ở train loss 8.65 với 1.8% val char accuracy — vẫn nằm sâu trong
plateau blank của CTC, **chưa từng phát ra một ký tự nào**. Lần full escape plateau
quanh step 125–150 và đạt đỉnh gần step 275. Notebook này dùng **300 step cho mọi
điểm**, đủ vượt plateau ở mọi điểm và đưa điểm cuối vào đúng vùng đã hội tụ.

> **Không còn mốc cross-check.** Con số 92.31% cho `n=250` đo ở local thuộc về bộ
> nhãn *trước* lần sửa 538 ảnh, nên không so sánh được với lần chạy này. Lần chạy
> Colab này chính là mốc mới; ghi lại `curve.json` của nó để đối chiếu về sau.


## 1. GPU + version

In [ ]:
!nvidia-smi -L || echo "NO GPU -- the whole curve is CPU-bound and will be slow"
import sys, torch, platform
print("python", sys.version.split()[0], "|", platform.platform())
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"

## 2. Lấy project code

Giống `colab_train.ipynb`: **A.** `git clone`, hoặc **B.** upload một zip **chỉ chứa
code** (`captcha/`, `out/`, `configs/`, `requirements*.txt`) — **đừng** kèm `pass/`.

`out/learning_curve.py` bắt buộc phải có: notebook gọi lại chính module đó thay vì
chép lại logic, nên local và Colab chạy đúng một định nghĩa của thí nghiệm.

In [ ]:
import os, zipfile, tarfile
PROJECT = "/content/scan-captcha-project"

# Chạy ô này -> Colab hiện nút "Chọn tệp" / "Choose Files".
# Bấm vào đó rồi chọn code.zip ở ổ C (55 KB).
from google.colab import files
up = files.upload()
name = next(iter(up))
os.makedirs(PROJECT, exist_ok=True)

# code.zip từng được đóng gói nhầm thành tar (đuôi .zip nhưng ruột là tar) và
# zipfile.ZipFile ném BadZipFile. Nhận dạng theo nội dung, không theo đuôi file.
if zipfile.is_zipfile(name):
    with zipfile.ZipFile(name) as z:
        z.extractall(PROJECT)
elif tarfile.is_tarfile(name):
    with tarfile.open(name) as t:
        t.extractall(PROJECT)
else:
    raise SystemExit(f"{name} không phải zip cũng không phải tar")

print(sorted(os.listdir(PROJECT))[:20])
assert os.path.isdir(f"{PROJECT}/captcha"), f"chưa có code ở {PROJECT}"
assert os.path.isfile(f"{PROJECT}/out/learning_curve.py"),     "thiếu out/learning_curve.py trong zip code"

# --- Cách khác: git clone (không cần nếu đã upload code.zip ở trên) --------
# !git clone https://github.com/<user>/<repo>.git {PROJECT}


## 3. Cài dependencies

In [ ]:
%cd {PROJECT}
!pip install -q -r requirements.txt -r requirements-optional.txt
!python -c "import torch, cv2, yaml, onnxruntime; print('deps OK')"

## 4. Lấy dataset

`env.zip` (233 MB, chính là thư mục `pass/` đã zip lại) phải nằm **sẵn trong My Drive**
**trước khi** chạy ô dưới — ở gốc hay trong thư mục con đều được, ô dưới tự dò ra.
Mở `drive.google.com`, kéo thả `C:\Project\scan-captcha-project\env.zip` vào,
đợi upload xong 100%.

Đừng dùng `files.upload()` cho file này — 233 MB qua widget của Colab rất chậm và
đứt giữa chừng là mất sạch; qua Drive thì upload một lần, các session sau dùng lại.

> **Phải upload đủ bộ, đừng rút gọn còn ảnh 80×30.** Ngưỡng tần suất của charset
> được tính trên **toàn bộ index**, nên bỏ bớt các nhóm khác sẽ đổi luôn tập bị loại
> của `g_80x30`. Đã thấy đúng hiệu ứng đó: 538 ảnh nhóm khác được đổi nhãn làm
> `train` dịch từ 1562 sang 1563. Lệch 1 ảnh = 0.5% trên 195 ảnh test.


In [ ]:
DATA = "/content/pass"          # thư mục chứa 25,976 file .png

# Chạy ô này -> Colab xin quyền truy cập Drive: bấm link hiện ra, chọn tài khoản
# Google, bấm "Cho phép" / "Allow".
from google.colab import drive
drive.mount("/content/drive")

import os, glob, shutil

# env.zip có thể nằm ở gốc My Drive hoặc trong một thư mục con -- link chia sẻ chỉ
# cho ra folder ID, không cho ra tên thư mục, nên dò thay vì hardcode đường dẫn.
hits = sorted(glob.glob("/content/drive/MyDrive/**/env.zip", recursive=True), key=len)
assert hits, ("không thấy env.zip trong My Drive. Nếu file nằm ở 'Shared with me', "
              "mở Drive -> chuột phải file -> 'Add shortcut to Drive' vào My Drive.")
ZIP_IN_DRIVE = hits[0]
print("dùng:", ZIP_IN_DRIVE,
      f"({round(os.path.getsize(ZIP_IN_DRIVE) / 1e6)} MB)")

!unzip -q -o "{ZIP_IN_DRIVE}" -d /content/_data

# env.zip gói sẵn một thư mục `pass/` bên trong. Giải thẳng vào /content/pass sẽ
# ra /content/pass/pass/..., nên tìm đúng thư mục chứa PNG rồi mới đặt vào chỗ.
cands = [d for d, _, fs in os.walk("/content/_data")
         if any(f.lower().endswith(".png") for f in fs)]
assert cands, "không thấy PNG nào trong env.zip"
src = min(cands, key=len)
if os.path.isdir(DATA):
    shutil.rmtree(DATA)
shutil.move(src, DATA)
shutil.rmtree("/content/_data", ignore_errors=True)

n = len([f for f in os.listdir(DATA) if f.lower().endswith(".png")])
print(f"{DATA}: {n} PNG")
assert n == 25976, f"expected 25976 images, found {n}"


## 5. Chạy đường cong

Bốn điểm, mỗi điểm 300 optimizer step. Trên T4 nhanh hơn CPU local rất nhiều —
toàn bộ curve thường xong trong khoảng 15 phút.

`curve.json` được ghi **sau mỗi điểm**, không phải chỉ ở cuối, nên nếu Colab
disconnect giữa chừng thì các điểm đã xong vẫn còn.

In [ ]:
import sys
from pathlib import Path

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
sys.path.insert(0, f"{PROJECT}/out")

# Pre-flight. Một index cache cũ có thể âm thầm đổi split. Đã gặp hai lần ở máy
# local: một cache trả về 25,374 record thay vì 25,976; và một cache dựng trước khi
# 538 ảnh được đổi nhãn vẫn được coi là hợp lệ (signature chỉ gồm số file + tổng
# byte + mtime mới nhất, mà rename không đổi cái nào) nên cho train=1562 trong khi
# scan thật cho 1563. Colab luôn scan mới, nên 1563 là con số đúng ở đây.
from captcha.config import Config, default_config_path
from captcha.dataset import scan_index, prepare_group

_cfg = Config.load(default_config_path(Path(PROJECT)))
_recs, _ = scan_index(Path(DATA))
_gd = prepare_group(_recs, _cfg.by_id("g_80x30"), _cfg.train)
print(f"index={len(_recs)}  train={len(_gd.train)} "
      f"val={len(_gd.val)} test={len(_gd.test)}")
assert len(_recs) == 25976, f"index={len(_recs)}, expected 25976 -- stale .cache?"
assert (len(_gd.train), len(_gd.val), len(_gd.test)) == (1563, 195, 195), \
    "split khác mong đợi -- dataset đã đổi, dừng lại kiểm tra trước khi chạy curve"

import learning_curve as lc
lc.main(group="g_80x30", device_name="cuda", data_dir=Path(DATA), workers=2)

## 6. Vẽ đường cong

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt

rows = json.loads((Path(PROJECT) / "out/learning_curve/curve.json").read_text())
xs = [r["n_train"] for r in rows]

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(xs, [r["test_em"] * 100 for r in rows], "o-", label="exact-match")
ax.plot(xs, [r["test_char"] * 100 for r in rows], "s--", label="per-character")
ax.set_xlabel("training images")
ax.set_ylabel("% on the fixed held-out test split")
ax.set_title(f"{lc.GROUP} learning curve (~{lc.STEPS} optimizer steps per point)")
ax.set_ylim(0, 101)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

for r in rows:
    print(f"{r['n_train']:5d} imgs ({r['steps']:4d} steps, best epoch "
          f"{r['best_epoch']:3d}) -> test EM {r['test_em']:6.2%}  "
          f"char {r['test_char']:6.2%}  CER {r['test_cer']:.4f}")

## 7. Đọc kết quả

Đoạn cuối là chỗ quyết định. Đọc nó cùng với **trần nhiễu nhãn ~96–97%** đã biết
cho nhóm này, và nhớ rằng per-character (~99%) mới là chỉ số phản ánh chất lượng
model thật — exact-match bị chặn bởi nhãn sai, không phải bởi model.

In [ ]:
CEILING = 0.965          # ~96.5%: trần do nhiễu nhãn, đo được ở lần full run

prev, last = rows[-2], rows[-1]
gain = (last["test_em"] - prev["test_em"]) * 100
ratio = last["n_train"] / prev["n_train"]
headroom = (CEILING - last["test_em"]) * 100

print(f"{ratio:.2f}x dữ liệu ({prev['n_train']} -> {last['n_train']}) "
      f"đổi exact-match {gain:+.2f} điểm")
print(f"còn cách trần nhiễu nhãn ~{headroom:.2f} điểm\n")

if gain < 1.0:
    print("=> Đường cong đã phẳng trước khi dùng hết dữ liệu.")
    print("   Thêm ảnh cùng nguồn sẽ không dịch được exact-match;")
    print("   thứ còn lại để sửa là NHÃN, không phải số lượng ảnh.")
else:
    print(f"=> Vẫn còn leo ({gain:+.2f} điểm cho {ratio:.2f}x dữ liệu).")
    print("   Thêm ảnh cùng nguồn vẫn có ích, nhưng chú ý trần ~96.5%:")
    print("   phần lớn dư địa còn lại là nhãn sai, không phải thiếu ảnh.")

## 8. Tải kết quả về

In [ ]:
import shutil
from pathlib import Path

src = Path(PROJECT) / "out/learning_curve"
shutil.make_archive("/content/learning_curve", "zip", src)
print("ZIP:", round(Path("/content/learning_curve.zip").stat().st_size / 1e6, 1), "MB")

try:
    from google.colab import files
    files.download("/content/learning_curve.zip")
except Exception as e:
    print("tải thủ công:", e)